In [1]:
pip install gradio   #can use for graphical user interface

In [2]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 49.6 MB/s eta 0:00:00


In [6]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
import gradio as gr
import PIL.Image as Image
import numpy as np
import cv2
from ultralytics import YOLO

# Load YOLO model
model = YOLO("/content/drive/MyDrive/CRISPR/best.pt")

def extract_channel(img_np, channels):
    """Extract specified color channels (list) with black background."""
    filtered = np.zeros_like(img_np)
    channel_idx_map = {"red": 0, "green": 1, "blue": 2}  # OpenCV uses BGR order
    for ch in channels:
        idx = channel_idx_map[ch]
        filtered[:, :, idx] = img_np[:, :, idx]
    return filtered

def calculate_rlu(img_np, boxes, channels):
    """Calculate RLU inside circular droplet masks for given color channels."""
    rlu_values = []
    annotated_img = img_np.copy()

    # Channel mapping
    channel_idx_map = {"red": 0, "green": 1, "blue": 2}
    idxs = [channel_idx_map[ch] for ch in channels]

    for box in boxes:
        x1, y1, x2, y2 = map(int, box)
        cropped = img_np[y1:y2, x1:x2].copy()

        h, w = cropped.shape[:2]
        mask = np.zeros((h, w), dtype=np.uint8)
        center = (w // 2, h // 2)
        radius = min(w, h) // 2
        cv2.circle(mask, center, radius, 1, -1)

        # Sum selected channels
        intensity = np.sum(cropped[:, :, idxs], axis=2)
        rlu = int(np.sum(intensity * mask))
        rlu_str = f"{rlu:.2e}"
        rlu_values.append(rlu_str)

        # Overlay droplet outline and label
        if set(channels) == {"red", "green"}:
            color = (0, 255, 255)  # yellow outline for red+green
        else:
            color = (255, 0, 0)  # blue outline
        cv2.circle(annotated_img, (x1 + center[0], y1 + center[1]), radius, color, 2)
        cv2.putText(
            annotated_img,
            f"RFU: {rlu_str}",
            (x1, max(30, y1 - 10)),
            cv2.FONT_HERSHEY_SIMPLEX,
            1.5,
            (255, 255, 255),
            5,
            cv2.LINE_AA,
        )

    return annotated_img, rlu_values

def predict_image(img, conf_threshold, iou_threshold):
    """YOLO detection + RFU measurement for combined red+green and blue channels."""
    results = model.predict(
        source=img,
        conf=conf_threshold,
        iou=iou_threshold,
        show_labels=True,
        show_conf=True,
        imgsz=640,
    )

    img_np = np.array(img)

    # Extract red+green combined and blue-only channels
    red_green_img = extract_channel(img_np, ["red", "green"])
    blue_img = extract_channel(img_np, ["blue"])

    combined_rlu, blue_rlu = [], []
    annotated_red_green, annotated_blue = None, None

    for r in results:
        boxes = r.boxes.xyxy.cpu().numpy()

        annotated_red_green, combined_rlu = calculate_rlu(red_green_img, boxes, ["red", "green"])
        annotated_blue, blue_rlu = calculate_rlu(blue_img, boxes, ["blue"])

        im_array = r.plot()
        yolo_annotated = Image.fromarray(im_array[..., ::-1])

    return (
        yolo_annotated,
        combined_rlu,
        blue_rlu,
        Image.fromarray(annotated_red_green),
        Image.fromarray(annotated_blue),
    )

# --- Gradio Interface ---
iface = gr.Interface(
    fn=predict_image,
    inputs=[
        gr.Image(type="pil", label="Upload Image"),
        gr.Slider(minimum=0, maximum=1, value=0.25, label="Confidence threshold"),
        gr.Slider(minimum=0, maximum=1, value=0.45, label="IoU threshold"),
    ],
    outputs=[
        gr.Image(type="pil", label="YOLO Detection Result"),
        gr.Textbox(label="Red + Green Channel RFU (scientific notation)"),
        gr.Textbox(label="Blue Channel RFU (scientific notation)"),
        gr.Image(type="pil", label="Red + Green Channel with RFU Overlay"),
        gr.Image(type="pil", label="Blue Channel with RFU Overlay"),
    ],
    title="Fluorescence RFU Detection (Red+Green vs Blue)",
    description="Detect droplets using YOLO and calculate RFU for combined red+green fluorescence and blue fluorescence channels.",
)

if __name__ == "__main__":
    iface.launch(share=True)


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://89e9e6eabe8fe14f2b.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
